In [1]:
# ── BLOCK 1: Imports & Load PaySim ──
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, accuracy_score,
                             roc_auc_score, classification_report)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import json, os, warnings
warnings.filterwarnings('ignore')

# Load PaySim
print("Loading PaySim dataset...")
df_pay = pd.read_csv(
    "../data/PS_20174392719_1491204439457_log.csv")

print(f"Full dataset shape: {df_pay.shape}")
print(f"Columns: {list(df_pay.columns)}")
print(f"\nTransaction types:")
print(df_pay['type'].value_counts())
print(f"\nFraud rate (full): {df_pay['isFraud'].mean()*100:.4f}%")

Loading PaySim dataset...
Full dataset shape: (6362620, 11)
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Fraud rate (full): 0.1291%


In [2]:
# ── BLOCK 2: Filter & Explore PaySim ──

# Keep only TRANSFER and CASH_OUT — only types with fraud
df_pay_filtered = df_pay[
    df_pay['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

print(f"After filtering to TRANSFER & CASH_OUT:")
print(f"Shape: {df_pay_filtered.shape}")
print(f"Fraud rate: {df_pay_filtered['isFraud'].mean()*100:.4f}%")
print(f"Fraud cases: {df_pay_filtered['isFraud'].sum():,}")
print()

# Amount statistics
print("=== Amount Statistics ===")
print(df_pay_filtered['amount'].describe())
print()

# Fraud by type
print("=== Fraud by Transaction Type ===")
for t in ['TRANSFER', 'CASH_OUT']:
    subset = df_pay_filtered[df_pay_filtered['type'] == t]
    frauds = subset['isFraud'].sum()
    rate   = subset['isFraud'].mean() * 100
    print(f"{t}: {frauds:,} frauds / {len(subset):,} "
          f"total ({rate:.3f}%)")

After filtering to TRANSFER & CASH_OUT:
Shape: (2770409, 11)
Fraud rate: 0.2965%
Fraud cases: 8,213

=== Amount Statistics ===
count    2.770409e+06
mean     3.175361e+05
std      8.877897e+05
min      0.000000e+00
25%      8.297354e+04
50%      1.712609e+05
75%      3.067912e+05
max      9.244552e+07
Name: amount, dtype: float64

=== Fraud by Transaction Type ===
TRANSFER: 4,097 frauds / 532,909 total (0.769%)
CASH_OUT: 4,116 frauds / 2,237,500 total (0.184%)


In [3]:
# ── BLOCK 3: Risk Matrix + Prepare Features ──

# Percentile-based risk matrix
p25 = df_pay_filtered['amount'].quantile(0.25)
p50 = df_pay_filtered['amount'].quantile(0.50)
p75 = df_pay_filtered['amount'].quantile(0.75)

print("=== PaySim Risk Matrix Thresholds ===")
print(f"Low:      $0 – ${p25:,.0f}        (0-25th percentile)")
print(f"Medium:   ${p25:,.0f} – ${p50:,.0f}  (25-50th percentile)")
print(f"High:     ${p50:,.0f} – ${p75:,.0f}  (50-75th percentile)")
print(f"Critical: above ${p75:,.0f}           (75th+ percentile)")
print()

def assign_risk_tier_pay(amount):
    if amount <= p25:
        return 'Low'
    elif amount <= p50:
        return 'Medium'
    elif amount <= p75:
        return 'High'
    else:
        return 'Critical'

df_pay_filtered['Risk_Tier'] = df_pay_filtered['amount'].apply(
    assign_risk_tier_pay)

print("=== Fraud Rate by Risk Tier ===")
for tier in ['Low','Medium','High','Critical']:
    subset     = df_pay_filtered[df_pay_filtered['Risk_Tier']==tier]
    frauds     = subset['isFraud'].sum()
    total      = len(subset)
    fraud_rate = (frauds/total)*100
    avg_amt    = subset['amount'].mean()
    print(f"{tier:10s}: {frauds:5,} frauds / {total:7,} total "
          f"({fraud_rate:.3f}%) | Avg: ${avg_amt:,.0f}")

print()

# Prepare features
# Drop non-numeric and ID columns
features = ['amount', 'oldbalanceOrg', 'newbalanceOrig',
            'oldbalanceDest', 'newbalanceDest', 'step']

# Add transaction type as numeric
df_pay_filtered['type_num'] = (
    df_pay_filtered['type'] == 'TRANSFER').astype(int)
features.append('type_num')

X_pay = df_pay_filtered[features].copy()
y_pay = df_pay_filtered['isFraud']
rt_pay = df_pay_filtered['Risk_Tier']

print(f"Features used: {features}")
print(f"Dataset size: {X_pay.shape}")

=== PaySim Risk Matrix Thresholds ===
Low:      $0 – $82,974        (0-25th percentile)
Medium:   $82,974 – $171,261  (25-50th percentile)
High:     $171,261 – $306,791  (50-75th percentile)
Critical: above $306,791           (75th+ percentile)

=== Fraud Rate by Risk Tier ===
Low       : 1,516 frauds / 692,603 total (0.219%) | Avg: $42,096
Medium    :   987 frauds / 692,602 total (0.143%) | Avg: $125,646
High      :   955 frauds / 692,602 total (0.138%) | Avg: $231,241
Critical  : 4,755 frauds / 692,602 total (0.687%) | Avg: $871,162

Features used: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'step', 'type_num']
Dataset size: (2770409, 7)


In [4]:
# ── BLOCK 4: Prepare PaySim for Modelling ──

# Scale features
scaler_pay = StandardScaler()
X_pay_scaled = pd.DataFrame(
    scaler_pay.fit_transform(X_pay),
    columns=X_pay.columns)

print("Splitting data...")
X_tr, X_te, y_tr, y_te, rt_tr, rt_te = train_test_split(
    X_pay_scaled, y_pay, rt_pay,
    test_size=0.2, random_state=42, stratify=y_pay)

print(f"Training set: {X_tr.shape}")
print(f"Test set:     {X_te.shape}")
print(f"Fraud in train: {y_tr.sum():,}")
print(f"Fraud in test:  {y_te.sum():,}")
print()

# SMOTE
print("Applying SMOTE — this may take a few minutes...")
smote = SMOTE(random_state=42)
X_tr_bal, y_tr_bal = smote.fit_resample(X_tr, y_tr)

print(f"After SMOTE: {X_tr_bal.shape}")
print(f"Fraud cases: {y_tr_bal.sum():,}")
print()

# Convert to tensors
X_tr_tensor = torch.FloatTensor(X_tr_bal.values)
y_tr_tensor = torch.LongTensor(y_tr_bal.values)
X_te_tensor = torch.FloatTensor(X_te.values)
y_te_tensor = torch.LongTensor(y_te.values)

# DataLoaders
tr_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
tr_loader  = DataLoader(tr_dataset, batch_size=512, shuffle=True)

print("✅ Data ready!")

Splitting data...
Training set: (2216327, 7)
Test set:     (554082, 7)
Fraud in train: 6,570
Fraud in test:  1,643

Applying SMOTE — this may take a few minutes...
After SMOTE: (4419514, 7)
Fraud cases: 2,209,757

✅ Data ready!


In [5]:
# ── BLOCK 5: Define Models (seeded) ──
import random
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

class L2DNetwork(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super(L2DNetwork, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.classifier_head = nn.Linear(32, num_classes)
        self.deferral_head   = nn.Linear(32, 1)

    def forward(self, x):
        shared_out   = self.shared(x)
        class_logits = self.classifier_head(shared_out)
        defer_prob   = torch.sigmoid(self.deferral_head(shared_out))
        return class_logits, defer_prob

def l2d_loss(class_logits, defer_prob, targets, cost):
    ce_loss    = nn.CrossEntropyLoss(reduction='none')
    class_loss = ce_loss(class_logits, targets)
    defer_p    = defer_prob.squeeze()
    return ((1-defer_p)*class_loss + defer_p*cost).mean()

RISK_WEIGHTS = {
    'Low': 1.0, 'Medium': 0.8,
    'High': 0.5, 'Critical': 0.2}

def rs_l2d_loss(class_logits, defer_prob,
                targets, risk_tier_batch, base_cost):
    ce_loss    = nn.CrossEntropyLoss(reduction='none')
    class_loss = ce_loss(class_logits, targets)
    defer_p    = defer_prob.squeeze()
    weights    = torch.tensor(
        [RISK_WEIGHTS[t] for t in risk_tier_batch],
        dtype=torch.float32)
    instance_cost = weights * base_cost
    return ((1-defer_p)*class_loss + defer_p*instance_cost).mean()

class FraudDatasetWithTiers(torch.utils.data.Dataset):
    def __init__(self, X, y, tiers):
        self.X = X; self.y = y; self.tiers = tiers
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.tiers[idx]

# Seeded DataLoader so batch order is reproducible
tr_loader = DataLoader(tr_dataset, batch_size=512, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))

input_dim = X_tr_tensor.shape[1]
print(f"{check} All defined (seeded)! Input dimensions: {input_dim}")


✅ All defined (seeded)! Input dimensions: 7


In [6]:
# ── BLOCK 6: Full Cost Calibration Sweep ──
# The quick 5-epoch sweeps above (cost 0.1-0.5) hinted at instability. A systematic,
# non-early-abort sweep across a much wider range (0.13 to 1.0) was run separately
# (see scratchpad script run_paysim_calibrate.py; raw results in
# implementation/results/paysim_cost_calibration.json). Summary of that sweep:
#
#   cost=0.13  -> COLLAPSED_HIGH (99.63% deferral after 1 epoch)
#   cost=0.2   -> COLLAPSED_HIGH (99.54% deferral after 1 epoch)
#   cost=0.3   -> COLLAPSED_LOW  (0.00% deferral by epoch 8)
#   cost=0.4   -> trending upward, still climbing at epoch 8 (0.29%) when the
#                 early-abort sweep cut it off -- the only candidate NOT yet collapsed
#   cost=0.5, 0.65, 0.8, 1.0 -> all COLLAPSED_LOW (0.00% deferral by epoch 8)
#
# PaySim's response to base cost is bimodal on this feature-poor (7-feature) dataset:
# almost every tested value either defers ~everything or ~nothing. cost=0.4 was the
# only value showing a genuine intermediate trajectory, so it was run separately for
# the full, un-aborted 15 epochs (see scratchpad script run_paysim_cost04_full.py) to
# confirm it actually stabilises rather than eventually collapsing too. It does --
# see Block 7 below.
PAY_BASE_COST = 0.4
print(f"Selected base cost for PaySim: {PAY_BASE_COST} "
      f"(only value found to reach a stable, non-degenerate operating point)")


Selected base cost for PaySim: 0.4 (only value found to reach a stable, non-degenerate operating point)


In [7]:
# ── BLOCK 7: Train All 3 Models (seeded, cost=0.4, full 15 epochs) ──
torch.manual_seed(SEED)
print("=" * 50); print("Model 1 -- Baseline (deferral disabled via extreme cost)"); print("=" * 50)
baseline_pay = L2DNetwork(input_dim=input_dim)
opt1 = optim.Adam(baseline_pay.parameters(), lr=0.001)
for epoch in range(15):
    baseline_pay.train()
    for X_b, y_b in tr_loader:
        opt1.zero_grad()
        cl, dp = baseline_pay(X_b)
        loss = l2d_loss(cl, dp, y_b, cost=0.99)
        loss.backward()
        opt1.step()
print(f"{check} Baseline done!")

torch.manual_seed(SEED)
print("=" * 50); print(f"Model 2 -- Standard L2D (cost={PAY_BASE_COST})"); print("=" * 50)
std_pay = L2DNetwork(input_dim=input_dim)
opt2 = optim.Adam(std_pay.parameters(), lr=0.001)
for epoch in range(15):
    std_pay.train()
    epoch_loss, nb_ = 0.0, 0
    for X_b, y_b in tr_loader:
        opt2.zero_grad()
        cl, dp = std_pay(X_b)
        loss = l2d_loss(cl, dp, y_b, cost=PAY_BASE_COST)
        loss.backward()
        opt2.step()
        epoch_loss += loss.item(); nb_ += 1
    print(f"  Epoch {epoch+1:2d}/15 | Loss: {epoch_loss/nb_:.4f}")
print(f"{check} Standard L2D done!")

rt_tr_values  = rt_tr.values
original_size = len(rt_tr_values)
smote_size    = len(X_tr_bal) - original_size
rt_tr_bal     = list(rt_tr_values) + ['Low'] * smote_size
rs_dataset_pay = FraudDatasetWithTiers(X_tr_tensor, y_tr_tensor, rt_tr_bal)
rs_loader_pay  = DataLoader(rs_dataset_pay, batch_size=512, shuffle=True,
                            generator=torch.Generator().manual_seed(SEED))

torch.manual_seed(SEED)
print("=" * 50); print(f"Model 3 -- Risk-Sensitive L2D (base_cost={PAY_BASE_COST})"); print("=" * 50)
rs_pay = L2DNetwork(input_dim=input_dim)
opt3 = optim.Adam(rs_pay.parameters(), lr=0.001)
for epoch in range(15):
    rs_pay.train()
    epoch_loss, nb_ = 0.0, 0
    for X_b, y_b, t_b in rs_loader_pay:
        opt3.zero_grad()
        cl, dp = rs_pay(X_b)
        loss = rs_l2d_loss(cl, dp, y_b, list(t_b), PAY_BASE_COST)
        loss.backward()
        opt3.step()
        epoch_loss += loss.item(); nb_ += 1
    print(f"  Epoch {epoch+1:2d}/15 | Loss: {epoch_loss/nb_:.4f}")
print(f"{check} Risk-Sensitive L2D done!")


Model 1 -- Baseline (deferral disabled via extreme cost)
✅ Baseline done!
Model 2 -- Standard L2D (cost=0.4)
  Epoch  1/15 | Loss: 0.0557
  Epoch  5/15 | Loss: 0.0233
  Epoch 10/15 | Loss: 0.0215
  Epoch 15/15 | Loss: 0.0212
✅ Standard L2D done!
Model 3 -- Risk-Sensitive L2D (base_cost=0.4)
  Epoch  1/15 | Loss: 0.0967
  Epoch  5/15 | Loss: 0.0218
  Epoch 10/15 | Loss: 0.0198
  Epoch 15/15 | Loss: 0.0192
✅ Risk-Sensitive L2D done!


In [9]:
# ── BLOCK 8: Evaluate All 3 Models on PaySim ──
def evaluate_pay(model, X_te, y_te, rt_te, model_name):
    model.eval()
    with torch.no_grad():
        cl, dp     = model(X_te)
        cp         = torch.argmax(cl, dim=1).numpy()
        dp         = dp.squeeze().numpy()
        deferred   = dp > 0.5
        autonomous = ~deferred
        y_np       = y_te.values
        rt_np      = rt_te.values
    print(f"\n{'='*50}\n  {model_name}\n{'='*50}")
    print(f"Deferred:   {deferred.sum():,} ({deferred.mean()*100:.2f}%)")
    print(f"Autonomous: {autonomous.sum():,} ({autonomous.mean()*100:.2f}%)")
    result = {'deferral_rate': round(float(deferred.mean()*100), 2)}
    if autonomous.sum() > 0:
        f1  = f1_score(y_np[autonomous], cp[autonomous], zero_division=0)
        acc = accuracy_score(y_np[autonomous], cp[autonomous])
        auc = roc_auc_score(y_np[autonomous], cl[autonomous].numpy()[:, 1])
        print(f"F1 Score:   {f1:.4f} | Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")
        fraud_def   = int(deferred[y_np==1].sum())
        fraud_miss  = int(((cp[autonomous]==0) & (y_np[autonomous]==1)).sum())
        fraud_catch = int(((cp[autonomous]==1) & (y_np[autonomous]==1)).sum())
        print(f"Frauds deferred: {fraud_def} | caught: {fraud_catch} | missed: {fraud_miss}")
    print("Deferral by Risk Tier:")
    tier_rates = {}
    for tier in ['Low','Medium','High','Critical']:
        mask = rt_np == tier
        rate = round(float(deferred[mask].mean() * 100), 2)
        tier_rates[tier] = rate
        print(f"  {tier:10s}: {rate:.2f}%")
    return tier_rates

r1 = evaluate_pay(baseline_pay, X_te_tensor, y_te, rt_te, "MODEL 1 -- BASELINE")
r2 = evaluate_pay(std_pay,      X_te_tensor, y_te, rt_te, "MODEL 2 -- STANDARD L2D (cost=0.4)")
r3 = evaluate_pay(rs_pay,       X_te_tensor, y_te, rt_te, "MODEL 3 -- RISK-SENSITIVE L2D (base_cost=0.4)")

print(f"\n{'='*50}\nKEY COMPARISON -- CRITICAL-TIER DEFERRAL\n{'='*50}")
ratio = r3['Critical'] / r2['Critical'] if r2['Critical'] > 0 else float('inf')
print(f"Model 2 Critical: {r2['Critical']:.2f}% | Model 3 Critical: {r3['Critical']:.2f}% "
      f"| Ratio: {ratio:.2f}x")


  MODEL 1 -- BASELINE
Deferred:   0 (0.00%)
Autonomous: 554,082 (100.00%)
F1 Score:   0.6477 | Accuracy: 0.9968 | ROC-AUC: 0.9984
Frauds deferred: 0 | caught: 1630 | missed: 13
Deferral by Risk Tier:
  Low       : 0.00%
  Medium    : 0.00%
  High      : 0.00%
  Critical  : 0.00%

  MODEL 2 -- STANDARD L2D (cost=0.4)
Deferred:   2,496 (0.45%)
Autonomous: 551,586 (99.55%)
F1 Score:   0.5994 | Accuracy: 0.9960 | ROC-AUC: 0.9985
Frauds deferred: 1 | caught: 1633 | missed: 9
Deferral by Risk Tier:
  Low       : 0.00%
  Medium    : 0.44%
  High      : 0.69%
  Critical  : 0.68%

  MODEL 3 -- RISK-SENSITIVE L2D (base_cost=0.4)
Deferred:   7,079 (1.28%)
Autonomous: 547,003 (98.72%)
F1 Score:   0.6169 | Accuracy: 0.9963 | ROC-AUC: 0.9985
Frauds deferred: 2 | caught: 1632 | missed: 9
Deferral by Risk Tier:
  Low       : 0.02%
  Medium    : 0.51%
  High      : 0.74%
  Critical  : 3.84%

KEY COMPARISON -- CRITICAL-TIER DEFERRAL
Model 2 Critical: 0.68% | Model 3 Critical: 3.84% | Ratio: 5.65x


In [10]:
import json
pay_results = {
    'dataset': 'PaySim',
    'base_cost': PAY_BASE_COST,
    'risk_weights': RISK_WEIGHTS,
    'seed': SEED,
    'note': ('Base cost recalibrated from 0.13 to 0.4 after a full calibration sweep found the '
             "model's response to cost is bimodal on this 7-feature dataset -- see Block 6 above "
             'and implementation/results/paysim_cost_calibration.json.'),
    'model1_baseline': r1,
    'model2_standard_l2d': r2,
    'model3_risk_sensitive': r3,
    'critical_tier_m2_to_m3_ratio': round(r3['Critical'] / r2['Critical'], 2) if r2['Critical'] > 0 else None,
}
with open("../results/paysim_results.json", 'w') as f:
    json.dump(pay_results, f, indent=4)
print("✅ Saved!")


✅ Saved!
